# ADSC x AIAA — Deliverable 1: Predicting Julian Dates with XGBoost, ARIMA & Ensemble

**ADSC Fall 2026 | AIAA Project**

---

## Overview

In this deliverable you will build **three predictive models** on real astronomical data:

| Model | Type | What it learns |
|-------|------|----------------|
| **XGBoost** | Supervised regression | Predict `julian` from orbital features |
| **ARIMA** | Time-series | Forecast `julian` as a sequential signal |
| **Ensemble** | Meta-model | Weighted combination of XGBoost + ARIMA |

### Dataset
**Solar System Planets Motions** — Kaggle  
🔗 [https://www.kaggle.com/datasets/cpitrat/solar-system-planets-motions/data](https://www.kaggle.com/datasets/cpitrat/solar-system-planets-motions/data)

The dataset records the heliocentric positions, velocities, and orbital parameters of solar-system bodies sampled at discrete **Julian Dates** (`julian`). Your goal: given the orbital state of the planets, predict the corresponding Julian Date.

### Why Julian Date?
A [Julian Date (JD)](https://en.wikipedia.org/wiki/Julian_day) is a continuous day-count used by astronomers. It is the most compact, timezone-agnostic representation of time in astrophysics. Predicting JD from planetary positions is the classical **"where are the planets?"** problem inverted — think of it as letting the model solve its own clock.

---

## Learning Objectives

By the end of this deliverable you should be able to:

1. Load, clean, and explore a real-world dataset with pandas.
2. Engineer lag features and rolling statistics for time-aware ML.
3. Train and evaluate an **XGBoost** regressor.
4. Diagnose a time series with ACF/PACF plots and fit an **ARIMA** model.
5. Build a simple **ensemble** by blending model outputs.
6. Interpret model performance with standard regression metrics.

---

## Cells marked `# TODO`

Each `# TODO` block calls out exactly what you need to implement. The surrounding scaffold — imports, variable names, helper functions — is already written. Focus on filling in the missing pieces rather than rewriting from scratch.


## Resources

Keep these tabs open while working through the notebook.

### XGBoost
- 📺 **StatQuest — XGBoost Clearly Explained (Part 1: Regression)**  
  [https://www.youtube.com/watch?v=OtD8wVaFm6E](https://www.youtube.com/watch?v=OtD8wVaFm6E)  
  Great intuition for how gradient boosting builds trees iteratively on residuals.
- 📺 **StatQuest — XGBoost Part 2 (Mathematical Details)**  
  [https://www.youtube.com/watch?v=8b1JEDvenQU](https://www.youtube.com/watch?v=8b1JEDvenQU)
- 📖 **XGBoost Python API docs**  
  [https://xgboost.readthedocs.io/en/stable/python/python_api.html](https://xgboost.readthedocs.io/en/stable/python/python_api.html)

### ARIMA / Time Series
- 📺 **Ritvikmath — ARIMA Model Explained**  
  [https://www.youtube.com/watch?v=vqzO-9LSoG4](https://www.youtube.com/watch?v=vqzO-9LSoG4)  
  Step-by-step breakdown of the AR, I, and MA components.
- 📺 **StatQuest — Autocorrelation & ACF Plots**  
  [https://www.youtube.com/watch?v=ZjaBn93YPWo](https://www.youtube.com/watch?v=ZjaBn93YPWo)
- 📖 **statsmodels ARIMA docs**  
  [https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html)

### Ensemble Methods
- 📺 **StatQuest — Ensemble Methods (Bagging & Boosting)**  
  [https://www.youtube.com/watch?v=2Mg8QD0F1dQ](https://www.youtube.com/watch?v=2Mg8QD0F1dQ)
- 📖 **sklearn Model Evaluation**  
  [https://scikit-learn.org/stable/modules/model_evaluation.html](https://scikit-learn.org/stable/modules/model_evaluation.html)

### Pandas / Data Wrangling
- 📺 **Corey Schafer — Python Pandas Tutorial (full playlist)**  
  [https://www.youtube.com/playlist?list=PL-osiE80TeTsWmV9i9c58mdDCSskIFdDS](https://www.youtube.com/playlist?list=PL-osiE80TeTsWmV9i9c58mdDCSskIFdDS)
- 📖 **Pandas 10-minute guide**  
  [https://pandas.pydata.org/docs/user_guide/10min.html](https://pandas.pydata.org/docs/user_guide/10min.html)


---

## Section 0 — Environment Setup

Install missing packages once, then restart the kernel.

```bash
pip install xgboost statsmodels scikit-learn pandas numpy matplotlib seaborn
```


In [ ]:
# ── Core data libraries ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Machine learning ──────────────────────────────────────────────────────────
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Time-series ────────────────────────────────────────────────────────────────
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── Misc ───────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.titlesize': 13})
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
print('All imports OK')

---

## Section 1 — Data Loading

### Downloading the dataset

1. Create a free [Kaggle](https://www.kaggle.com) account if you don't have one.
2. Go to the dataset page:  
   🔗 [https://www.kaggle.com/datasets/cpitrat/solar-system-planets-motions/data](https://www.kaggle.com/datasets/cpitrat/solar-system-planets-motions/data)
3. Click **Download** and place the CSV(s) in a `data/` folder next to this notebook.

**Optional — Kaggle API (faster):**
```bash
pip install kaggle
# place kaggle.json in ~/.kaggle/
kaggle datasets download -d cpitrat/solar-system-planets-motions --unzip -p data/
```

### Column glossary (expected)

| Column | Description |
|--------|-------------|
| `julian` | **Target.** Julian Date — the continuous day-count since noon 1 Jan 4713 BC |
| `body` | Planet / body name (Mercury, Venus, Earth, …) |
| `x`, `y`, `z` | Heliocentric ecliptic coordinates (AU) |
| `vx`, `vy`, `vz` | Velocity components (AU/day) |
| `lighttime` | One-way light travel time to the body (days) |
| `range_` | Distance from the Sun (AU) |
| `elong` | Solar elongation angle (°) |

> The exact column names may differ slightly from what is shown above. Always run `df.columns` and `df.head()` first.


In [ ]:
import os

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

# ── Load the CSV ──────────────────────────────────────────────────────────────
# If the dataset unpacks into multiple files, concatenate them here.
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
print('CSV files found:', csv_files)

# TODO (1/1): Replace 'planets.csv' with the actual filename(s) you downloaded.
#             If there are multiple CSVs, use pd.concat([...]) to join them.
df = pd.read_csv(os.path.join(DATA_DIR, csv_files[0]))  # adjust as needed

print(f'Shape: {df.shape}')
df.head()

---

## Section 2 — Exploratory Data Analysis (EDA)

> **Goal:** understand what the data looks like before touching models.  
> Reference: 📺 [Pandas Tutorial Playlist](https://www.youtube.com/playlist?list=PL-osiE80TeTsWmV9i9c58mdDCSskIFdDS)


In [ ]:
# ── Shape, dtypes, missing values ─────────────────────────────────────────────
print('=== Info ===')
df.info()
print('\n=== Missing values ===')
print(df.isnull().sum())
print('\n=== Basic statistics ===')
df.describe()

In [ ]:
# ── Distribution of the target variable ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['julian'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Julian Date (target)')
axes[0].set_xlabel('Julian Date')
axes[0].set_ylabel('Count')

# TODO (2/1): Plot a box-plot of `julian` grouped by planet/body name.
#             Hint: use sns.boxplot(data=df, x='body', y='julian', ax=axes[1])
#             Does the julian range differ by body?
axes[1].set_title('TODO: Julian Date by Body')

plt.tight_layout()
plt.show()

In [ ]:
# ── Time-series view: pick one planet and plot its coordinates over time ──────
# We sort by julian so the time axis makes sense.
df_sorted = df.sort_values('julian').reset_index(drop=True)

# Filter to a single body for clarity — change 'Earth' if not in the dataset
FOCUS_BODY = df_sorted['body'].iloc[0] if 'body' in df_sorted.columns else None
if FOCUS_BODY:
    body_df = df_sorted[df_sorted['body'] == FOCUS_BODY].copy()
    print(f'Focusing on: {FOCUS_BODY}  ({len(body_df)} rows)')
else:
    body_df = df_sorted.copy()

# Plot x-coordinate over Julian Date
if 'x' in body_df.columns:
    plt.figure(figsize=(14, 4))
    plt.plot(body_df['julian'], body_df['x'], linewidth=0.8, color='tomato')
    plt.title(f'{FOCUS_BODY} — x-coordinate over time')
    plt.xlabel('Julian Date')
    plt.ylabel('x (AU)')
    plt.tight_layout()
    plt.show()

# TODO (2/2): Add plots for the y and z coordinates on the same or separate figures.
#             Do the coordinates show periodic (cyclical) behaviour? What does that tell you
#             about whether ARIMA might be appropriate?

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
numeric_cols = df_sorted.select_dtypes(include=[np.number]).columns.tolist()
corr = df_sorted[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# TODO (2/3): Which features are most correlated with `julian`?
#             Which features are highly correlated with *each other* (multicollinear)?
#             Write a short comment below summarising your findings.

# YOUR ANALYSIS:

---

## Section 3 — Feature Engineering

Raw features alone are often not enough. We'll create:

- **Lag features** — the value of a column *k* steps back in time (gives the model memory).
- **Rolling statistics** — moving averages and standard deviations (smooth out noise).
- **Encoded categoricals** — convert the `body` string to an integer for XGBoost.

> 📺 Lag features explained in context of time series ML:  
> [https://www.youtube.com/watch?v=d4Sn6ny_5LI](https://www.youtube.com/watch?v=d4Sn6ny_5LI)


In [ ]:
# ── Encode the body (planet) column ───────────────────────────────────────────
df_feat = df_sorted.copy()

if 'body' in df_feat.columns:
    le = LabelEncoder()
    df_feat['body_enc'] = le.fit_transform(df_feat['body'])
    print('Body encoding map:', dict(zip(le.classes_, le.transform(le.classes_))))
else:
    print('No body column found — skipping encoding')

In [ ]:
# ── Lag features ──────────────────────────────────────────────────────────────
# We compute lags within each body so we don't bleed information across planets.
LAG_COLS = [c for c in ['x', 'y', 'z', 'vx', 'vy', 'vz'] if c in df_feat.columns]
LAG_STEPS = [1, 2, 3]

group_key = 'body' if 'body' in df_feat.columns else None

for col in LAG_COLS:
    for lag in LAG_STEPS:
        new_col = f'{col}_lag{lag}'
        if group_key:
            df_feat[new_col] = df_feat.groupby(group_key)[col].shift(lag)
        else:
            df_feat[new_col] = df_feat[col].shift(lag)

print(f'Created {len(LAG_COLS) * len(LAG_STEPS)} lag features')
print(df_feat.filter(like='_lag').head())

In [ ]:
# TODO (3/1): Create rolling-window statistics for at least two columns.
#
# Example for x with a 5-step window:
#   df_feat['x_roll_mean5'] = df_feat.groupby('body')['x'].transform(
#       lambda s: s.rolling(5, min_periods=1).mean()
#   )
#   df_feat['x_roll_std5'] = df_feat.groupby('body')['x'].transform(
#       lambda s: s.rolling(5, min_periods=1).std()
#   )
#
# Do the same for at least one velocity column (vx, vy, or vz).
# Then print df_feat.shape to confirm the new columns were added.

# YOUR CODE HERE
raise NotImplementedError('TODO (3/1): Add rolling statistics — remove this line when done')

In [ ]:
# ── Drop rows with NaN introduced by lags/rolling ────────────────────────────
before = len(df_feat)
df_feat.dropna(inplace=True)
print(f'Dropped {before - len(df_feat)} rows with NaN. Remaining: {len(df_feat)}')

# Define feature matrix X and target y
drop_cols = ['julian', 'body'] if 'body' in df_feat.columns else ['julian']
X = df_feat.drop(columns=drop_cols)
y = df_feat['julian']

print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print('Features:', X.columns.tolist())

---

## Section 4 — Train / Test Split

**Important:** for time-series data we must NOT shuffle the split — future rows must never leak into the training set.  
We use `shuffle=False` and take the last 20 % of rows as the test set.


In [ ]:
TEST_SIZE = 0.2

# Time-ordered split — no shuffle!
split_idx = int(len(X) * (1 - TEST_SIZE))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train julian range: [{y_train.min():.1f}, {y_train.max():.1f}]')
print(f'Test  julian range: [{y_test.min():.1f}, {y_test.max():.1f}]')

---

## Section 5 — Model 1: XGBoost Regressor

XGBoost builds an ensemble of **decision trees** where each new tree corrects the residual errors of all previous trees (gradient boosting). It is one of the most widely used algorithms for tabular data.

### Resources
- 📺 **StatQuest — XGBoost Part 1 (Regression)**  
  [https://www.youtube.com/watch?v=OtD8wVaFm6E](https://www.youtube.com/watch?v=OtD8wVaFm6E)
- 📺 **StatQuest — XGBoost Part 2 (Math Details)**  
  [https://www.youtube.com/watch?v=8b1JEDvenQU](https://www.youtube.com/watch?v=8b1JEDvenQU)
- 📖 **Key hyperparameters:**  
  `n_estimators` — number of trees  
  `max_depth` — maximum tree depth (controls overfitting)  
  `learning_rate` (alias `eta`) — shrinks each tree's contribution  
  `subsample` — fraction of training rows used per tree  
  `colsample_bytree` — fraction of features used per tree  
  `reg_alpha`, `reg_lambda` — L1/L2 regularisation terms


In [ ]:
# ── Baseline XGBoost model ────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb   = r2_score(y_test, y_pred_xgb)

print(f'XGBoost Baseline — RMSE: {rmse_xgb:.4f}  |  MAE: {mae_xgb:.4f}  |  R²: {r2_xgb:.4f}')

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────────
feat_imp = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
feat_imp_top = feat_imp.nlargest(15)

plt.figure(figsize=(10, 6))
feat_imp_top.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('XGBoost — Top 15 Feature Importances (gain)')
plt.xlabel('Importance score')
plt.tight_layout()
plt.show()

In [ ]:
# TODO (5/1): Hyperparameter tuning.
#
# The baseline uses default-ish hyperparameters. Your task is to try at least
# THREE different hyperparameter configurations and compare their RMSE on the
# test set. Fill in the dictionary below and add more entries.
#
# Reference: https://xgboost.readthedocs.io/en/stable/parameter.html

configs = {
    'baseline': dict(n_estimators=200, max_depth=6,  learning_rate=0.10, subsample=0.8),
    'deep':     dict(n_estimators=300, max_depth=8,  learning_rate=0.05, subsample=0.8),
    # TODO: add at least one more configuration
}

results = {}
for name, params in configs.items():
    m = xgb.XGBRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    m.fit(X_train, y_train, verbose=False)
    preds = m.predict(X_test)
    results[name] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'MAE':  mean_absolute_error(y_test, preds),
        'R2':   r2_score(y_test, preds)
    }

print(pd.DataFrame(results).T.sort_values('RMSE'))

# TODO (5/2): Which configuration performed best? Store the best model in
#             `best_xgb` for use in the ensemble section.
#             best_xgb = ???

raise NotImplementedError('TODO (5/1 & 5/2): Complete hyperparameter tuning — remove this line when done')

In [ ]:
# TODO (5/3): Residual analysis.
#
# Plot the residuals (y_test - y_pred_xgb) against the predicted values.
# A well-calibrated model should show residuals randomly scattered around 0
# with no obvious pattern.
#
# 1. Compute residuals = y_test.values - y_pred_xgb
# 2. Scatter plot: x = y_pred_xgb, y = residuals
# 3. Draw a horizontal dashed line at y=0
# 4. Label axes and title the plot

# YOUR CODE HERE
raise NotImplementedError('TODO (5/3): Residual plot — remove this line when done')

---

## Section 6 — Model 2: ARIMA Time Series

**ARIMA(p, d, q)** models a univariate time series as a combination of:

| Component | Parameter | Meaning |
|-----------|-----------|--------|
| **AR** (Autoregressive) | `p` | The series is regressed on its own past `p` values |
| **I** (Integrated) | `d` | Number of differencing steps to make the series stationary |
| **MA** (Moving Average) | `q` | Error terms from the past `q` steps are included |

We will apply ARIMA to the `julian` values for a single planet (the model works best on a single, continuous univariate sequence).

### Resources
- 📺 **Ritvikmath — ARIMA Explained**  
  [https://www.youtube.com/watch?v=vqzO-9LSoG4](https://www.youtube.com/watch?v=vqzO-9LSoG4)
- 📺 **StatQuest — Autocorrelation (ACF)**  
  [https://www.youtube.com/watch?v=ZjaBn93YPWo](https://www.youtube.com/watch?v=ZjaBn93YPWo)
- 📖 **statsmodels ARIMA**  
  [https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html)


In [ ]:
# ── Extract the julian series for a single body ───────────────────────────────
if 'body' in df_feat.columns:
    arima_body = df_feat['body'].value_counts().idxmax()  # most common body
    julian_series = (
        df_feat[df_feat['body'] == arima_body]
        .sort_values('julian')['julian']
        .reset_index(drop=True)
    )
else:
    julian_series = df_feat['julian'].reset_index(drop=True)
    arima_body = 'all'

print(f'ARIMA body: {arima_body}  |  Series length: {len(julian_series)}')
print(julian_series.head(10))

In [ ]:
# ── Augmented Dickey-Fuller test for stationarity ─────────────────────────────
# A stationary series has constant mean and variance over time.
# ADF null hypothesis: the series has a unit root (non-stationary).
# Reject H0 (p < 0.05) → series is stationary → d = 0.

def adf_report(series, name='series'):
    result = adfuller(series.dropna())
    print(f'{name}:')
    print(f'  ADF statistic : {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.4f}')
    stationary = result[1] < 0.05
    print(f'  Conclusion    : {"Stationary (reject H0)" if stationary else "Non-stationary (fail to reject H0)"}')
    return stationary

is_stationary = adf_report(julian_series, 'Julian series (raw)')

if not is_stationary:
    diff1 = julian_series.diff().dropna()
    print()
    adf_report(diff1, 'Julian series (first difference)')

In [ ]:
# ── ACF and PACF plots ────────────────────────────────────────────────────────
# Use these to choose p and q:
#  - PACF cuts off after lag p  → set AR order to p
#  - ACF  cuts off after lag q  → set MA order to q
#
# Reference:
#   📺 StatQuest ACF: https://www.youtube.com/watch?v=ZjaBn93YPWo

series_for_acf = julian_series.diff().dropna() if not is_stationary else julian_series

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(series_for_acf,  lags=30, ax=axes[0], title='ACF — choose q (MA order)')
plot_pacf(series_for_acf, lags=30, ax=axes[1], title='PACF — choose p (AR order)')
plt.tight_layout()
plt.show()

print('Look at the plots above and decide your (p, d, q) values.')
print('d = 0 if ADF said stationary, d = 1 if we needed one differencing step.')

In [ ]:
# TODO (6/1): Choose your ARIMA orders based on the ACF/PACF plots above
#             and the ADF test result, then fit the model.
#
# Guidelines:
#   p — PACF cuts off after lag p (try p = 1 or 2 to start)
#   d — 0 if series is stationary, 1 if first difference was needed
#   q — ACF cuts off after lag q (try q = 0 or 1 to start)
#
# Steps:
#   1. Assign your chosen values below
#   2. Fit the ARIMA model on the training portion of julian_series
#   3. Print model.summary()

# Determine train/test lengths for the series
n_arima = len(julian_series)
n_arima_test = int(n_arima * TEST_SIZE)
arima_train = julian_series[: n_arima - n_arima_test]
arima_test  = julian_series[n_arima - n_arima_test :]

# TODO: set these values after examining the plots
p = None  # AR order
d = None  # Integration order
q = None  # MA order

if None in (p, d, q):
    raise NotImplementedError('TODO (6/1): Set p, d, q values — remove this line when done')

arima_model = ARIMA(arima_train, order=(p, d, q))
arima_fit   = arima_model.fit()
print(arima_fit.summary())

In [ ]:
# TODO (6/2): Generate forecasts for the test period.
#
# Use arima_fit.forecast(steps=n_arima_test) to produce forecasts.
# Then:
#   1. Compute RMSE and MAE against arima_test
#   2. Plot the training series, test actuals, and forecast on one figure
#   3. Shade the forecast confidence interval if available
#
# Hint:
#   forecast = arima_fit.forecast(steps=n_arima_test)
#   forecast_ci = arima_fit.get_forecast(steps=n_arima_test).conf_int()

# YOUR CODE HERE
raise NotImplementedError('TODO (6/2): ARIMA forecast and plot — remove this line when done')

# After completing, save predictions to this variable for the ensemble:
# y_pred_arima = ...  (numpy array of length n_arima_test)

---

## Section 7 — Ensemble Model

An ensemble combines the outputs of multiple models. The simplest form is a **weighted average**:

$$\hat{y}_{\text{ensemble}} = \alpha \cdot \hat{y}_{\text{XGBoost}} + (1 - \alpha) \cdot \hat{y}_{\text{ARIMA}}$$

where $\alpha \in [0, 1]$ is a blending weight you choose (or optimise).

The intuition: XGBoost is good at capturing **non-linear feature interactions**, while ARIMA is good at capturing **temporal autocorrelation**. The ensemble benefits from both.

### Resources
- 📺 **StatQuest — Ensemble Methods**  
  [https://www.youtube.com/watch?v=2Mg8QD0F1dQ](https://www.youtube.com/watch?v=2Mg8QD0F1dQ)
- 📺 **Ritvikmath — Stacking Ensembles**  
  [https://www.youtube.com/watch?v=8T2emza6g80](https://www.youtube.com/watch?v=8T2emza6g80)

> **Note:** XGBoost was trained on all planets; ARIMA was trained only on `{arima_body}`.  
> For a fair ensemble, restrict both predictions to the same subset.


In [ ]:
# ── Align XGBoost and ARIMA predictions to the same index ─────────────────────
# The two models may have been trained on different subsets.
# We need arrays of equal length that correspond to the same time steps.

# XGBoost predictions on the test set (already computed)
xgb_preds_aligned = y_pred_xgb.copy()   # shape (n_test,)

# ARIMA predictions (set in the previous cell)
# arima_preds_aligned = y_pred_arima.copy()  # uncomment after completing TODO 6/2

# Actuals for whichever subset we use
actuals_aligned = y_test.values.copy()   # shape (n_test,)

print('XGBoost preds shape :', xgb_preds_aligned.shape)
# print('ARIMA   preds shape :', arima_preds_aligned.shape)  # uncomment later
print('Actuals shape       :', actuals_aligned.shape)

In [ ]:
# TODO (7/1): Implement the weighted ensemble.
#
# 1. Try at least five alpha values: [0.1, 0.3, 0.5, 0.7, 0.9]
# 2. For each alpha compute the ensemble prediction and its RMSE.
# 3. Plot RMSE vs. alpha to find the best blending weight.
# 4. Compare the best ensemble RMSE to the standalone XGBoost and ARIMA RMSEs.
#
# Template:
#   alphas = [0.1, 0.3, 0.5, 0.7, 0.9]
#   rmse_list = []
#   for alpha in alphas:
#       ensemble_pred = alpha * xgb_preds_aligned + (1 - alpha) * arima_preds_aligned
#       rmse_list.append(np.sqrt(mean_squared_error(actuals_aligned, ensemble_pred)))
#   # plot alphas vs rmse_list

# YOUR CODE HERE
raise NotImplementedError('TODO (7/1): Implement and evaluate the ensemble — remove this line when done')

In [ ]:
# TODO (7/2) [STRETCH]: Optimise the blending weight automatically.
#
# Instead of a manual grid search, use scipy.optimize.minimize_scalar to find
# the alpha that minimises RMSE on the test set.
#
# from scipy.optimize import minimize_scalar
#
# def ensemble_rmse(alpha):
#     pred = alpha * xgb_preds_aligned + (1 - alpha) * arima_preds_aligned
#     return np.sqrt(mean_squared_error(actuals_aligned, pred))
#
# result = minimize_scalar(ensemble_rmse, bounds=(0, 1), method='bounded')
# best_alpha = result.x
# print(f'Optimal alpha: {best_alpha:.4f}  |  RMSE: {result.fun:.4f}')

# YOUR CODE HERE (optional)

---

## Section 8 — Model Evaluation & Comparison

We compare all three models on three standard regression metrics:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y - \hat{y})^2}$ | Average error in the same units as `julian` — penalises large errors |
| **MAE** | $\frac{1}{n}\sum|y - \hat{y}|$ | Average absolute error — more robust to outliers |
| **R²** | $1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}$ | Fraction of variance explained (1 = perfect, 0 = predicts mean) |

📖 sklearn metrics reference: [https://scikit-learn.org/stable/modules/model_evaluation.html](https://scikit-learn.org/stable/modules/model_evaluation.html)


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
def metrics(y_true, y_hat, name):
    return {
        'Model': name,
        'RMSE':  np.sqrt(mean_squared_error(y_true, y_hat)),
        'MAE':   mean_absolute_error(y_true, y_hat),
        'R2':    r2_score(y_true, y_hat)
    }

rows = [metrics(y_test, y_pred_xgb, 'XGBoost')]

# TODO (8/1): Uncomment and add ARIMA and Ensemble rows once those sections
#             are complete. Replace `y_pred_arima` and `y_pred_ensemble` with
#             the actual prediction arrays you produced above.
#
# rows.append(metrics(arima_test.values, y_pred_arima, 'ARIMA'))
# rows.append(metrics(actuals_aligned,   y_pred_ensemble, 'Ensemble'))

eval_df = pd.DataFrame(rows).set_index('Model')
print(eval_df.to_string())

In [ ]:
# TODO (8/2): Plot predicted vs. actual Julian Dates for the XGBoost model.
#
# 1. Scatter plot: x = y_test, y = y_pred_xgb
# 2. Add a perfect-prediction line (y = x, dashed)
# 3. Annotate with RMSE and R² values
# 4. Label axes: 'Actual Julian Date' and 'Predicted Julian Date'

# YOUR CODE HERE
raise NotImplementedError('TODO (8/2): Predicted vs actual plot — remove this line when done')

In [ ]:
# ── Time-series view of XGBoost predictions ───────────────────────────────────
# Sort by actual Julian Date to get a coherent time-ordered plot
order = np.argsort(y_test.values)
sorted_actual = y_test.values[order]
sorted_xgb    = y_pred_xgb[order]

plt.figure(figsize=(14, 5))
plt.plot(sorted_actual, label='Actual',    color='steelblue',  linewidth=1.5)
plt.plot(sorted_xgb,    label='XGBoost',   color='tomato',     linewidth=1.0, alpha=0.85)
plt.title('XGBoost — Predicted vs Actual Julian Dates (test set)')
plt.xlabel('Sample index (sorted by actual JD)')
plt.ylabel('Julian Date')
plt.legend()
plt.tight_layout()
plt.show()

# TODO (8/3): Add the ARIMA forecast line and the ensemble forecast line
#             to this same plot once those sections are complete.

In [ ]:
# TODO (8/4): Once all three models are complete, create a grouped bar chart
#             comparing RMSE, MAE, and R² side-by-side for XGBoost, ARIMA,
#             and the Ensemble.
#
# Hint:
#   eval_df[['RMSE', 'MAE']].plot(kind='bar', figsize=(10,5))
#   plt.title('Model Comparison — RMSE & MAE')

# YOUR CODE HERE
raise NotImplementedError('TODO (8/4): Model comparison bar chart — remove this line when done')

---

## Section 9 — Conclusion

**TODO (9/1):** Replace this placeholder with a 3–5 sentence summary of your findings. Address:

1. Which model (XGBoost, ARIMA, or Ensemble) performed best on RMSE? Why might that be?
2. What did the feature importance plot reveal about which orbital parameters are most predictive of Julian Date?
3. What were the limitations of the ARIMA model for this dataset?
4. How would you improve the ensemble if you had more time?

---

## Further Reading & Next Steps

If you want to go deeper after completing this deliverable:

- 📺 **Hyperparameter tuning with Optuna** — [https://www.youtube.com/watch?v=P6NwZVl8ttc](https://www.youtube.com/watch?v=P6NwZVl8ttc)
- 📺 **SARIMA and seasonal time series** — [https://www.youtube.com/watch?v=WjeGUs6mzXg](https://www.youtube.com/watch?v=WjeGUs6mzXg)
- 📖 **Prophet (Facebook's time series library)** — great for data with strong seasonality  
  [https://facebook.github.io/prophet/](https://facebook.github.io/prophet/)
- 📖 **SHAP values for model interpretability**  
  [https://shap.readthedocs.io/en/latest/](https://shap.readthedocs.io/en/latest/)
- 📺 **Cross-validation for time series (walk-forward)**  
  [https://www.youtube.com/watch?v=fmizNuFdCqo](https://www.youtube.com/watch?v=fmizNuFdCqo)
